<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/02_parse_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Document Loading and Parsing

## Phase 2 — Knowledge Base Ingestion

This notebook discovers and parses heterogeneous Northstar Commerce
knowledge sources.

Supported source types:

- SQL
- Markdown
- JSON
- YAML
- CSV

Ground-truth evaluation files are explicitly excluded from ingestion.

CSV operational tables are summarized rather than converted row-by-row
into RAG documents.

In [1]:
%pip install -q PyYAML

In [2]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import pandas as pd
import yaml

In [3]:
GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/metricguard-ai.git"
REPO_DIR = Path("/content/metricguard-ai")

In [4]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True
)

print("Repository cloned:", REPO_DIR)

Repository cloned: /content/metricguard-ai


In [5]:
# Define source locations

RAW_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"
GROUND_TRUTH_DIR = REPO_DIR / "data" / "ground_truth"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw data:", RAW_DIR)
print("Processed:", PROCESSED_DIR)
print("Ground truth:", GROUND_TRUTH_DIR)

Raw data: /content/metricguard-ai/data/raw
Processed: /content/metricguard-ai/data/processed
Ground truth: /content/metricguard-ai/data/ground_truth


In [6]:
# Define supported file types

SUPPORTED_EXTENSIONS = {
    ".sql",
    ".md",
    ".json",
    ".yml",
    ".yaml",
    ".csv"
}

In [7]:
# Discover source documents automatically

source_files = sorted(
    path
    for path in RAW_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print("Discovered source files:", len(source_files))

Discovered source files: 55


In [8]:
for path in source_files:
    print(path.relative_to(REPO_DIR))

data/raw/analyst_notes/active_customer_review.md
data/raw/analyst_notes/conversion_rate_migration.md
data/raw/analyst_notes/revenue_v3_migration.md
data/raw/analyst_notes/total_orders_semantics.md
data/raw/business_rules/active_customers/active_customers_v1.md
data/raw/business_rules/active_customers/active_customers_v2.md
data/raw/business_rules/conversion_rate/conversion_rate_v1.md
data/raw/business_rules/conversion_rate/conversion_rate_v2.md
data/raw/business_rules/net_revenue/net_revenue_v1.md
data/raw/business_rules/net_revenue/net_revenue_v2.md
data/raw/business_rules/net_revenue/net_revenue_v3.md
data/raw/business_rules/refund_rate/refund_rate_v1.md
data/raw/business_rules/refund_rate/refund_rate_v2.md
data/raw/business_rules/total_orders/total_orders_v1.md
data/raw/business_rules/total_orders/total_orders_v2.md
data/raw/dashboards/executive_kpi_dashboard.json
data/raw/dashboards/finance_revenue_dashboard.json
data/raw/dashboards/growth_marketing_dashboard.json
data/raw/dashboar

## SECURITY / LEAKAGE CHECK

In [9]:
ground_truth_leaks = [
    path
    for path in source_files
    if GROUND_TRUTH_DIR in path.parents
]

assert len(ground_truth_leaks) == 0, (
    f"Ground-truth leakage detected: {ground_truth_leaks}"
)

print("✅ Ground truth excluded from ingestion.")

✅ Ground truth excluded from ingestion.


## Inspect source-type distribution

In [10]:
extension_summary = (
    pd.Series(
        [path.suffix.lower() for path in source_files]
    )
    .value_counts()
    .rename_axis("extension")
    .reset_index(name="file_count")
)

extension_summary

,extension,file_count
0,.sql,23
1,.md,19
2,.csv,6
3,.json,4
4,.yml,3


## Create deterministic content hashing

In [11]:
def calculate_file_hash(path: Path) -> str:
    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(8192),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()

## Parser — SQL and Markdown

In [12]:
def parse_text_file(path: Path) -> dict:
    text = path.read_text(
        encoding="utf-8-sig"
    )

    return {
        "content": text,
        "structured_data": None
    }

## Parser — JSON

In [13]:
def parse_json_file(path: Path) -> dict:
    with path.open(
        "r",
        encoding="utf-8-sig"
    ) as file:
        data = json.load(file)

    normalized_text = json.dumps(
        data,
        indent=2,
        ensure_ascii=False
    )

    return {
        "content": normalized_text,
        "structured_data": data
    }

## Parser — YAML

In [14]:
def parse_yaml_file(path: Path) -> dict:
    with path.open(
        "r",
        encoding="utf-8-sig"
    ) as file:
        data = yaml.safe_load(file)

    normalized_text = yaml.safe_dump(
        data,
        sort_keys=False,
        allow_unicode=True
    )

    return {
        "content": normalized_text,
        "structured_data": data
    }

## Parser — CSV

In [15]:
def parse_csv_summary(path: Path) -> dict:
    df = pd.read_csv(path)

    metadata = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": list(df.columns),
        "dtypes": {
            column: str(dtype)
            for column, dtype in df.dtypes.items()
        },
        "null_counts": {
            column: int(count)
            for column, count in df.isna().sum().items()
        }
    }

    sample_rows = (
        df.head(3)
        .fillna("")
        .to_dict(orient="records")
    )

    summary_text = (
        f"Dataset: {path.stem}\n"
        f"Rows: {len(df)}\n"
        f"Columns: {len(df.columns)}\n"
        f"Column names: {', '.join(df.columns)}\n\n"
        f"Sample rows:\n"
        f"{json.dumps(sample_rows, indent=2, default=str)}"
    )

    return {
        "content": summary_text,
        "structured_data": metadata
    }

## Create source-type classifier

In [16]:
def get_source_type(path: Path) -> str:
    extension = path.suffix.lower()

    mapping = {
        ".sql": "sql",
        ".md": "markdown",
        ".json": "json",
        ".yml": "yaml",
        ".yaml": "yaml",
        ".csv": "csv"
    }

    return mapping[extension]

## Create parser dispatcher

In [17]:
def parse_file(path: Path) -> dict:
    extension = path.suffix.lower()

    if extension in {".sql", ".md"}:
        return parse_text_file(path)

    if extension == ".json":
        return parse_json_file(path)

    if extension in {".yml", ".yaml"}:
        return parse_yaml_file(path)

    if extension == ".csv":
        return parse_csv_summary(path)

    raise ValueError(
        f"Unsupported file type: {extension}"
    )

## Canonical parsed-document format

In [18]:
def build_parsed_document(path: Path) -> dict:
    parsed = parse_file(path)

    relative_path = path.relative_to(REPO_DIR)

    content_hash = calculate_file_hash(path)

    document_id = (
        f"{path.stem}-"
        f"{content_hash[:12]}"
    )

    return {
        "document_id": document_id,
        "source_path": str(relative_path),
        "file_name": path.name,
        "source_type": get_source_type(path),
        "content_hash": content_hash,
        "content": parsed["content"],
        "structured_data": parsed["structured_data"]
    }

## Parsing the entire Northstar knowledge base

In [19]:
parsed_documents = []
parse_errors = []

for path in source_files:

    try:
        document = build_parsed_document(path)
        parsed_documents.append(document)

    except Exception as exc:
        parse_errors.append({
            "source_path": str(
                path.relative_to(REPO_DIR)
            ),
            "error": str(exc)
        })

print("Successfully parsed:", len(parsed_documents))
print("Parse errors:", len(parse_errors))

Successfully parsed: 55
Parse errors: 0


In [20]:
parse_errors

[]

In [21]:
# Inspection example

parsed_documents[0]

{'document_id': 'active_customer_review-1f2c34aa6c80',
 'source_path': 'data/raw/analyst_notes/active_customer_review.md',
 'file_name': 'active_customer_review.md',
 'source_type': 'markdown',
 'content_hash': '1f2c34aa6c80438913880b4b4a3aab3fc5d5357eac702f6be42c79e4a314bb2e',
 'content': '# Active Customer Definition Review\n\nauthor: Rohan Mehta\nteam: Customer Analytics\ndate: 2026-03-10\nrelated_metric: active_customers\n\nThe enterprise Active Customer definition changed on March 1, 2026.\n\nThe approved definition now requires at least one successfully paid order\nduring the previous 30 days.\n\nGrowth reporting currently continues to count identified customers with recent\ndigital activity.\n\nThat measure remains useful for engagement analysis but should not be treated\nas equivalent to the enterprise Active Customers KPI.\n\nRecommendation:\n\nEither migrate the Growth dashboard to Active Customers v2 or rename the\nexisting measure to Digital Active Customers.\n',
 'structur

## Build Parsing Summary

In [22]:
parse_summary = pd.DataFrame(
    [
        {
            "document_id": doc["document_id"],
            "source_path": doc["source_path"],
            "source_type": doc["source_type"],
            "content_length": len(doc["content"])
        }
        for doc in parsed_documents
    ]
)

parse_summary.head()

,document_id,source_path,source_type,content_length
0,active_customer_review-1f2c34aa6c80,data/raw/analyst_notes/active_customer_review.md,markdown,673
1,conversion_rate_migration-71ee7fd90aa8,data/raw/analyst_notes/conversion_rate_migrati...,markdown,553
2,revenue_v3_migration-d802bb74c774,data/raw/analyst_notes/revenue_v3_migration.md,markdown,696
3,total_orders_semantics-92e0cd8e8726,data/raw/analyst_notes/total_orders_semantics.md,markdown,593
4,active_customers_v1-d3ddd1ffc374,data/raw/business_rules/active_customers/activ...,markdown,408


In [23]:
parse_summary.groupby(
    "source_type"
).agg(
    documents=("document_id", "count"),
    total_characters=("content_length", "sum")
)

,documents,total_characters
source_type,,
csv,6,5984
json,4,3290
markdown,19,10889
sql,23,14085
yaml,3,3587


## Validation

In [24]:
assert all(
    doc["content"].strip()
    for doc in parsed_documents
)

print("✅ No empty parsed documents.")

✅ No empty parsed documents.


In [25]:
document_ids = [
    doc["document_id"]
    for doc in parsed_documents
]

assert len(document_ids) == len(set(document_ids))

print("✅ All document IDs are unique.")

✅ All document IDs are unique.


In [26]:
assert not any(
    "ground_truth" in doc["source_path"]
    for doc in parsed_documents
)

print("✅ No evaluation leakage.")

✅ No evaluation leakage.


## Writing parsed documents as JSONL

In [27]:
OUTPUT_PATH = (
    PROCESSED_DIR /
    "parsed_documents.jsonl"
)

with OUTPUT_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    for document in parsed_documents:

        file.write(
            json.dumps(
                document,
                ensure_ascii=False,
                default=str
            )
            + "\n"
        )

print("Saved:", OUTPUT_PATH)

Saved: /content/metricguard-ai/data/processed/parsed_documents.jsonl


## Creating Parsing Manifest

In [28]:
MANIFEST_PATH = (
    PROCESSED_DIR /
    "parse_manifest.csv"
)

parse_summary.to_csv(
    MANIFEST_PATH,
    index=False
)

print("Saved:", MANIFEST_PATH)

Saved: /content/metricguard-ai/data/processed/parse_manifest.csv


## Final parser report

In [29]:
print("=" * 60)
print("METRICGUARD INGESTION REPORT")
print("=" * 60)

print(
    f"Source files discovered : {len(source_files)}"
)

print(
    f"Documents parsed        : {len(parsed_documents)}"
)

print(
    f"Parse errors            : {len(parse_errors)}"
)

print(
    f"Ground-truth leakage    : 0"
)

print(
    f"Output                  : {OUTPUT_PATH}"
)

METRICGUARD INGESTION REPORT
Source files discovered : 55
Documents parsed        : 55
Parse errors            : 0
Ground-truth leakage    : 0
Output                  : /content/metricguard-ai/data/processed/parsed_documents.jsonl
